# ADIM 3A - Sentetik eğitim verisi ve eğitim öncesi ölçüm

Ben bu defteri Colab T4 ortamında eğitim hazırlığı, veri doğrulaması ve eğitim öncesi/sonrası aynı değerlendirme kümesini kullanmak için çalıştırıyorum. Bu defter model indirmez, QLoRA eğitimi yapmaz, Hugging Face'e yükleme yapmaz ve dış API çağrısı yapmaz.

## Biçim varsayımım

Hocanın örnek Colab defterine erişimim olmadığı için JSONL satırlarında `messages` alanını kullanan OpenAI/TRL tarzı sohbet biçimini varsayıyorum. Bu varsayımı doğrulanmış uyumluluk olarak sunmuyorum. Örnek defter farklı bir şema istiyorsa yalnızca dönüştürme katmanını güncelleyeceğim; kaynak, ayrım ve ölçüm kontrollerini koruyacağım.

In [ ]:
from pathlib import Path
import sys

KOK_DIZIN = Path.cwd()
if not (KOK_DIZIN / 'egitim_verisi.py').exists():
    raise FileNotFoundError('Benim proje kök dizinimde egitim_verisi.py bulunmalı.')
if str(KOK_DIZIN) not in sys.path:
    sys.path.insert(0, str(KOK_DIZIN))

from egitim_verisi import (
    ARAC_SEMALARI,
    CIKTI_SABLONU_DOSYASI,
    DEGERLENDIRME_DOSYASI,
    degerlendirme_metriklerini_hesapla,
    jsonl_oku,
    veri_dogrula,
    veri_kumelerini_uret,
)


In [ ]:
sayilar = veri_kumelerini_uret()
rapor = veri_dogrula()
assert not rapor['hatalar'], rapor['hatalar']
assert sayilar['egitim'] >= 400
rapor


## Araç sözleşmem

Ben ajan örneklerinde yalnızca proje içindeki gerçek araç adlarını ve tek parametreli şemalarını kullanıyorum. `konaklama_ucreti` aracının tek parametresi JSON metnidir; bu JSON'un iç şeması araç kodunda doğrulanır.

In [ ]:
ARAC_SEMALARI


## Eğitim öncesi ölçümüm

Ben eğitim öncesi ve sonrası için aynı `degerlendirme.jsonl` sorularını kullanıyorum. Model çıktıları `degerlendirme_cikti_sablonu.json` biçiminde ayrı olarak kaydedilir. Metriklerim kimlik doğruluğu, cevap işareti doğruluğu, geçerli JSON, doğru araç ve RAG doğruluğudur. Cevap sonu işareti proje yapılandırmasında boş olduğu için bu metrik, işaret dayatılmadığını doğrular.

In [ ]:
degerlendirme_ornekleri = jsonl_oku(DEGERLENDIRME_DOSYASI)
print(f'Ben aynı ölçüm için {len(degerlendirme_ornekleri)} değerlendirme sorusu hazırladım.')
print(f'Ben model çıktısı şablonunu burada saklıyorum: {CIKTI_SABLONU_DOSYASI}')

# Ben GPU/model gerektiren çıkarımı bu ADIM 3A kapsamında çalıştırmıyorum.
bos_model_ciktilari = {}
degerlendirme_metriklerini_hesapla(bos_model_ciktilari)


## Gemini için ayrı ve kapalı hazırlık

Ben burada Gemini API çağrısı yapmıyorum. Gerçek dış servis üretimi gerekirse anahtarı yalnızca Colab Secrets veya ortam değişkeninden alacağım; anahtarı koda yazmayacağım. Çağrı yapılmadan önce ayrıca açık onay alacağım.

In [ ]:
import os

GEMINI_URETIMI_AKTIF = False
gemini_api_anahtari = os.environ.get('GEMINI_API_KEY')
if GEMINI_URETIMI_AKTIF:
    raise RuntimeError('Ben API çağrısını bu aşamada kapalı tutuyorum; önce açık onay gerekir.')
print('Ben Gemini üretimini kapalı tuttum; hiçbir dış API çağrısı yapılmadı.')
